In [1]:
def predict_and_save(
    con,
    model_path,
    relation="test",
    target="click",
    output_dir=None,
    n_rows=200_000,
    threshold=0.5,
    extract=True,
    seed=42,
):
    """Run a serialized model over a DuckDB relation and persist predictions.

    Loads the pipeline, draws a reproducible sample from ``relation``,
    computes the positive-class probability for each row, measures the
    inference time, and writes the true label and the predicted
    probability to a Parquet file. The output file is always named after
    the model with the suffix ``_predictions.parquet``. When
    ``extract=True``, one false positive and one false negative are
    returned in memory along with their raw feature values, so they can
    be fed directly to an explainer.

    Args:
        con (duckdb.DuckDBPyConnection): Active DuckDB connection.
        model_path (str or pathlib.Path): Path to the serialized
            pipeline in joblib format.
        relation (str): Name of the DuckDB view or table with the test
            data. Defaults to ``"test"``.
        target (str): Name of the binary target column. Defaults to
            ``"click"``.
        output_dir (str, pathlib.Path, or None): Directory where the
            Parquet file is written. The file is named after the model
            stem with the suffix ``_predictions.parquet``. When
            ``None``, the file is written next to the model. Defaults to
            ``None``.
        n_rows (int): Number of rows to draw from ``relation`` for the
            prediction set. When the value exceeds the total number of
            rows in the view, all rows are returned. Defaults to
            200,000.
        threshold (float): Decision threshold used to classify a row as
            a positive prediction. Defaults to 0.5.
        extract (bool): If ``True``, one false positive and one false
            negative are located and returned in memory. If ``False``,
            the extraction step is skipped and ``failing_cases`` is
            ``None``. Defaults to ``True``.
        seed (int): Seed for the reservoir sampler and for the failing
            case selection. Defaults to 42.

    Returns:
        dict: Dictionary with keys ``prediction_time`` (seconds),
        ``output_path`` (pathlib.Path), ``failing_cases``
        (pandas.DataFrame with two rows and the raw features, or
        ``None`` when ``extract=False``), ``n_failures``, ``n_fp`` and
        ``n_fn``.

    Raises:
        FileNotFoundError: If ``model_path`` does not exist.
        ValueError: If ``n_rows`` is not a positive integer, if
            ``threshold`` is not in ``(0, 1)``, or if the sample is
            empty.
    """
    import time
    
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    if not isinstance(n_rows, int) or isinstance(n_rows, bool) or n_rows <= 0:
        raise ValueError("'n_rows' must be a positive integer.")

    if not isinstance(threshold, (int, float)) or isinstance(threshold, bool) \
            or not (0 < threshold < 1):
        raise ValueError("'threshold' must be a number in (0, 1).")

    if not isinstance(extract, bool):
        raise TypeError("'extract' must be a boolean.")

    model = joblib.load(model_path)

    df = con.sql(f"""
        SELECT * FROM {relation}
        USING SAMPLE {n_rows} ROWS (reservoir, {seed})
    """).df()

    if df.empty:
        raise ValueError(f"The sample from '{relation}' returned no rows.")

    X = df.drop(columns=[target])
    y = df[target].astype("int8").to_numpy()

    start = time.perf_counter()
    proba = model.predict_proba(X)[:, 1]
    prediction_time = time.perf_counter() - start

    target_dir = Path(output_dir) if output_dir is not None else model_path.parent
    target_dir.mkdir(parents=True, exist_ok=True)
    output_path = target_dir / f"{model_path.stem}_predictions.parquet"

    pd.DataFrame({
        target: y,
        "prediction": proba.astype("float32"),
    }).to_parquet(output_path, index=False)

    y_pred = (proba >= threshold).astype("int8")

    n_failures = int((y != y_pred).sum())
    n_fp = int(((y_pred == 1) & (y == 0)).sum())
    n_fn = int(((y_pred == 0) & (y == 1)).sum())

    failing_cases = None
    if extract:
        df_full = df.copy()
        df_full["_pred"] = y_pred

        fp_pool = df_full[(df_full["_pred"] == 1) & (df_full[target] == 0)]
        fn_pool = df_full[(df_full["_pred"] == 0) & (df_full[target] == 1)]

        picks = []
        if not fp_pool.empty:
            picks.append(fp_pool.sample(1, random_state=seed))
        if not fn_pool.empty:
            picks.append(fn_pool.sample(1, random_state=seed + 1))

        if picks:
            failing_cases = pd.concat(picks, ignore_index=True)
        else:
            failing_cases = pd.DataFrame(columns=df_full.columns)

    return {
        "prediction_time": prediction_time,
        "output_path": output_path,
        "failing_cases": failing_cases,
        "n_failures": n_failures,
        "n_fp": n_fp,
        "n_fn": n_fn,
    }

In [2]:
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd


def build_lime_explainer(
    df_features,
    categorical_cols,
    numeric_cols,
    random_state=42,
):
    """Build a LIME explainer for a mixed categorical/numeric feature set.

    Categorical columns are encoded as integer codes before being handed
    to LIME, and the original category values are stored so that any
    perturbed array can be decoded back to the DataFrame format the
    pipeline expects. Numeric columns are preserved as floats with no
    intermediate string conversion.

    Args:
        df_features (pandas.DataFrame): Reference sample of raw feature
            values.
        categorical_cols (list of str): Names of the categorical columns.
        numeric_cols (list of str): Names of the numeric columns.
        random_state (int): Seed for the LIME sampler. Defaults to 42.

    Returns:
        tuple: ``(explainer, decode, columns, encoders)``.

    Raises:
        KeyError: If any name in ``categorical_cols`` or ``numeric_cols``
            is not a column of ``df_features``.
    """
    columns = list(df_features.columns)

    missing = [
        c for c in (categorical_cols + numeric_cols) if c not in columns
    ]
    if missing:
        raise KeyError(f"Columns not found in 'df_features': {missing}")

    original_dtypes = {col: df_features[col].dtype for col in columns}

    encoders = {}
    X_encoded = df_features.copy()

    for col in categorical_cols:
        cat = X_encoded[col].astype("category")
        encoders[col] = list(cat.cat.categories)
        X_encoded[col] = cat.cat.codes.astype(float)

    X_array = np.empty(X_encoded.shape, dtype=float)
    for i, col in enumerate(X_encoded.columns):
        X_array[:, i] = pd.to_numeric(
            X_encoded[col], errors="raise"
        ).to_numpy(dtype=float)

    cat_indices = [i for i, c in enumerate(columns) if c in encoders]
    cat_names = {i: encoders[columns[i]] for i in cat_indices}

    def decode(arr):
        """Map an encoded array back to a DataFrame matching the input.

        Categorical columns are restored to their original values and
        dtypes. Numeric columns keep their float values without any
        string conversion.

        Args:
            arr (numpy.ndarray): Array of shape ``(n, len(columns))``.

        Returns:
            pandas.DataFrame: DataFrame with the original column names
            and dtypes.
        """
        df = pd.DataFrame(arr, columns=columns)

        for col, cats in encoders.items():
            codes = df[col].round().astype(int).clip(0, len(cats) - 1)
            df[col] = pd.Series([cats[c] for c in codes], index=df.index)
            try:
                df[col] = df[col].astype(original_dtypes[col])
            except (ValueError, TypeError):
                pass

        for col in numeric_cols:
            try:
                df[col] = df[col].astype(original_dtypes[col])
            except (ValueError, TypeError):
                pass

        return df[columns]

    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_array,
        feature_names=columns,
        categorical_features=cat_indices,
        categorical_names=cat_names,
        mode="classification",
        discretize_continuous=True,
        random_state=random_state,
    )

    return explainer, decode, columns, encoders

In [3]:
import sys
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "test.parquet"
MODEL_PATH = PROJECT_ROOT / "models" / "sklearn"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from trainers import sklearn_trainer as skt
from utils import evaluation_toolkit as eva

con = duckdb.connect(database=":memory:")
con.execute(
    f"CREATE VIEW test AS SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')"
)
con.execute("PRAGMA disable_progress_bar")

mlp_100_50 = joblib.load(MODEL_PATH / "mlp_100_50_pipeline.joblib")
mlp_100 = joblib.load(MODEL_PATH / "mlp_100_pipeline.joblib")

In [4]:
PREDICTIONS_DIR = PROJECT_ROOT / "data" / "predictions"

result_100_50 = predict_and_save(
    con=con,
    model_path=MODEL_PATH / "mlp_100_50_pipeline.joblib",
    output_dir=PREDICTIONS_DIR,
    extract=True,
)

result_100 = predict_and_save(
    con=con,
    model_path=MODEL_PATH / "mlp_100_pipeline.joblib",
    output_dir=PREDICTIONS_DIR,
    extract=False,
)

In [5]:
fig, ax, (auc1, auc2) = eva.plot_roc_curves(
    pred1=result_100_50["output_path"],
    pred2=result_100["output_path"],
    model_names=("MLP (100, 50)", "MLP (100)"),
    title="Curvas ROC sobre el conjunto de prueba",
    xlabel="Tasa de falsos positivos",
    ylabel="Tasa de verdaderos positivos",
    figsize=(5, 5),
    colors=("black", "#888888"),
    pads=(14, 14, 14),
    tick_size=11, auc_decimals=3,
    label_size=12,
)

In [6]:
pred_100_50 = pd.read_parquet(result_100_50["output_path"])

youden_100_50 = eva.youden_optimal_threshold(
    y_true=pred_100_50["click"],
    y_score=pred_100_50["prediction"],
    model_name="MLP (100, 50)",
)

Model,Threshold,Sensitivity,Specificity,Youden Index
"MLP (100, 50)",.1570,.7745,.5916,.3661


In [7]:
pred_100 = pd.read_parquet(result_100["output_path"])

youden_100 = eva.youden_optimal_threshold(
    y_true=pred_100["click"],
    y_score=pred_100["prediction"],
    model_name="MLP (100)",
)

Model,Threshold,Sensitivity,Specificity,Youden Index
MLP (100),.1550,.7589,.6070,.3659


In [12]:
df_metrics = eva.metrics_comparison_table(
    sources=[result_100_50["output_path"], result_100["output_path"]],
    thresholds=[0.1570, 0.1550],
    model_names=["MLP (100, 50)", "MLP (100)"],
    metric_col_title="Métrica",  include_threshold=False,
    decimals=3,
    metric_titles={
        "threshold": "Umbral",
        "accuracy": "Exactitud",
        "precision": "Precisión",
        "recall": "Sensibilidad",
        "f1": "F1-score",
        "roc_auc": "AUC ROC",
    }
)

Métrica,"MLP (100, 50)",MLP (100)
Exactitud,.622,.633
Precisión,.278,.281
Sensibilidad,.774,.759
F1-score,.409,.410
AUC ROC,.745,.744


In [22]:
df_reference = (
    con.sql("""
        SELECT * FROM test
        USING SAMPLE 20000 ROWS (reservoir, 42)
    """)
    .df()
    .drop(columns=["click"])
)

explainer, decode, columns, encoders = build_lime_explainer(
    df_reference, categorical_cols, numeric_cols
)

# Caso a explicar
case = (
    result_100_50["failing_cases"]
    .query("_pred == 1 and click == 0")
    .iloc[0]
    .drop("_pred")
)

df_case = case[columns].to_frame().T

case_encoded = df_case.copy()
for col, cats in encoders.items():
    case_encoded[col] = cats.index(case[col])

case_array = case_encoded[columns].to_numpy(dtype=float)

# Verificación
df_dec = decode(case_array)

mismatches = []
for col in columns:
    if str(df_case[col].iloc[0]) != str(df_dec[col].iloc[0]):
        mismatches.append((col, df_case[col].iloc[0], df_dec[col].iloc[0]))

if mismatches:
    for col, o, d in mismatches:
        print(f"  {col:<25} orig={o!r:<30} dec={d!r}")
else:
    print("Sin discrepancias.")

orig = mlp_100_50.predict_proba(df_case)[0, 1]
dec = mlp_100_50.predict_proba(df_dec)[0, 1]
print(f"\noriginal = {orig:.10f}")
print(f"decoded  = {dec:.10f}")
print(f"diff     = {abs(orig - dec):.2e}")

Sin discrepancias.

original = 0.5179546475
decoded  = 0.5179546475
diff     = 0.00e+00


In [28]:
case_row = case_array.ravel()

explanation = explainer.explain_instance(
    case_row,
    predict_fn,
    num_features=10,
    num_samples=5000,
    labels=(1,),
)

for feature, weight in explanation.as_list(label=1):
    print(f"{feature:<45} {weight:>+10.4f}")

site_id=5b08c53b                                 +0.1618
C14=19016                                        +0.1264
site_domain=7687a86e                             +0.0880
log_freq_device_id <= 0.00                       +0.0537
C17=2162                                         -0.0488
app_domain=7801e8d9                              -0.0331
app_id=ecad2386                                  -0.0313
device_model=8571106b                            +0.0293
log_freq_device_ip <= 1.39                       +0.0291
C16=250                                          -0.0289
